# Chapter 6 · Building Self-Learning AI Agents

The chapter's learning loop, run against a real model. The agent writes an annual revenue memo, a grader checks the figure, a failure becomes a one-line lesson, and the lesson goes into a real `lessons.md` file that the agent reads before every later memo. Then the desk's notes are tidied while the agent is idle, and the agent is tested on a task it never learned from.

The chapter starts at [Building Self-Learning AI Agents](https://ashcastelinocs124.github.io/msba-ai-workshop/ch06-self-learning-agents.html). Each section of this notebook goes with one page:

- **Section 1 · The learning loop** goes with [page 6.2](https://ashcastelinocs124.github.io/msba-ai-workshop/ch06-learning-harness.html).
- **Section 2 · Dreaming and the playbook** goes with [page 6.3](https://ashcastelinocs124.github.io/msba-ai-workshop/ch06-memory-and-experience.html).
- **Section 3 · Held-out evaluation** goes with [page 6.4](https://ashcastelinocs124.github.io/msba-ai-workshop/ch06-for-reference.html).

Run the setup cell first, then the sections in order. Sections 2 and 3 use functions from section 1.

**Before you run:** the same three steps as chapter 1. Sign in at [lumen.ncsa.illinois.edu](https://lumen.ncsa.illinois.edu/chat), create an API key on your [profile page](https://lumen.ncsa.illinois.edu/profile), and add it to Colab as a secret named `LUMEN_API_KEY` with notebook access enabled.

The model is `glm-5.3-flash` on Lumen, the campus LLM service. A real model does not follow a script, so your run may not match the book's. Each section says what to look for.

In [ ]:
%pip install -q openai

In [ ]:
import json, os, re, shutil
from openai import OpenAI

try:
    from google.colab import userdata                 # on Colab: the key from Colab Secrets
    LUMEN_API_KEY = userdata.get("LUMEN_API_KEY")
except ImportError:                                   # anywhere else: an environment variable
    LUMEN_API_KEY = os.environ["LUMEN_API_KEY"]

MODEL = "glm-5.3-flash"  # highly preferred for this workshop
client = OpenAI(base_url="https://lumen.ncsa.illinois.edu/v1", api_key=LUMEN_API_KEY)

## Section 1 · The learning loop (page 6.2)

Monday to Thursday, the agent writes one annual revenue memo a day: Deere, Apple, Caterpillar, NVIDIA. The firm reports the **fiscal year** in each company's 10-K. The tool can also return the same twelve months on a **calendar-year** basis, added up from four quarterly reports. The two figures differ for every company except Caterpillar, whose fiscal year is the calendar year.

The loop has four steps around chapter 1's agent:

1. **Write.** The agent writes the memo, with `lessons.md` in its system prompt.
2. **Grade.** An automatic test checks that the memo's figure is the 10-K figure.
3. **Reflect.** After a failure, the model turns the grader's feedback into a one-line rule.
4. **Save.** The rule is added to `lessons.md`, under three rules of its own: only after a graded failure, never twice, and only with a person's approval.

First, the firm's records and the one read-only tool.

In [ ]:
# The firm's records, copied from the textbook's learning.py. Keep the two in step.
# Annual revenue as the firm's records hold it: the fiscal year in each company's 10-K, and the same twelve
# months added up from four quarterly reports on a calendar-year basis. Illustrative figures, not market data.
FILINGS = {
    "DE":   {"name": "Deere",       "fy": "FY2025 (ended Nov 2, 2025)",  "fiscal": 45.7e9,  "calendar": 44.9e9,  "page": 31},
    "AAPL": {"name": "Apple",       "fy": "FY2025 (ended Sep 27, 2025)", "fiscal": 416.2e9, "calendar": 425.4e9, "page": 28},
    "CAT":  {"name": "Caterpillar", "fy": "FY2025 (ended Dec 31, 2025)", "fiscal": 64.8e9,  "calendar": 64.8e9,  "page": 46},
    "NVDA": {"name": "NVIDIA",      "fy": "FY2026 (ended Jan 25, 2026)", "fiscal": 187.1e9, "calendar": 179.5e9, "page": 52},
    "MSFT": {"name": "Microsoft",   "fy": "FY2025 (ended Jun 30, 2025)", "fiscal": 281.7e9, "calendar": 296.3e9, "page": 37},
}
WEEK = [("Mon", "DE"), ("Tue", "AAPL"), ("Wed", "CAT"), ("Thu", "NVDA")]   # the practice tasks
HELD_OUT = ["MSFT"]                                                          # locked: never learned from (section 3)
PRIYA = "Firm uses fiscal year. Cite the 10-K."                              # her correction, by email
DESK = "You write revenue memos for Champaign Capital Research. Look up every figure with a tool before you write it."


def get_annual_revenue(ticker, basis="calendar"):
    """Annual revenue for a ticker, on a calendar-year or a fiscal-year basis, with where the figure comes from."""
    row = FILINGS.get(ticker.upper())
    if row is None:
        return {"error": f"no annual figures for {ticker}"}
    if basis == "fiscal":
        return {"name": row["name"], "period": row["fy"], "revenue": row["fiscal"], "source": f"10-K, page {row['page']}"}
    return {"name": row["name"], "period": "calendar 2025", "revenue": row["calendar"], "source": "four quarterly reports, added up"}


TOOLS = {"get_annual_revenue": get_annual_revenue}
TOOL_SCHEMAS = [{"type": "function", "function": {
    "name": "get_annual_revenue",
    "description": "Annual revenue for one company, on a calendar-year or a fiscal-year basis, with its source.",
    "strict": True,
    "parameters": {"type": "object",
                   "properties": {"ticker": {"type": "string", "enum": list(FILINGS)},
                                  "basis": {"type": "string", "enum": ["calendar", "fiscal"]}},
                   "required": ["ticker", "basis"], "additionalProperties": False}}}]


def billions(x):
    return f"${x / 1e9:.1f}B"


for t, row in FILINGS.items():
    print(f"{row['name']:<12} fiscal year {billions(row['fiscal'])} ({row['fy']}) · calendar 2025 {billions(row['calendar'])}")

The agent is chapter 1's loop with this one tool. It prints one `Step N:` line per step, like the textbook. The grader checks one thing: the memo's figure must equal the fiscal-year figure in the 10-K. A real model writes figures its own way, so the grader reads `$45.7B`, `$45.7 billion` and `$45,700,000,000` as the same figure.

In [ ]:
def _describe(name, args):
    return name.removeprefix("get_").replace("_", " ") + f" ({', '.join(str(v) for v in args.values())})"


def _describe_result(result):
    return ", ".join(f"{k}: {v:,.0f}" if isinstance(v, float) else f"{k}: {v}" for k, v in result.items())


def agent(question, system, max_steps=5, verbose=True):
    """Chapter 1's loop: ask the model, run any tool it calls, hand back the result, until it answers."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOL_SCHEMAS).choices[0].message
        if not msg.tool_calls:
            text = (msg.content or "").strip()
            if verbose:
                print(f"Step {step}: answered — {text}")
            return text
        msgs.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls]})
        for c in msg.tool_calls:
            args = json.loads(c.function.arguments or "{}")
            try:
                result = TOOLS[c.function.name](**args)
            except (KeyError, TypeError) as e:          # a tool that does not exist, or arguments it does not take
                result = {"error": f"{c.function.name} could not run: {e}"}
            if verbose:
                print(f"Step {step}: looked up {_describe(c.function.name, args)} → {_describe_result(result)}")
            msgs.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(result)})
    if verbose:
        print("Stopped: step budget exhausted.")
    return ""


# ponytail: reads $ figures with B, bn, billion, M or million, or written out in full; "$45.7" with no unit is read as $45.70
_MONEY = re.compile(r"(\$|USD\s?)?(\d[\d,]*(?:\.\d+)?)\s*(billion|bn|B|million|M)?\b", re.I)
_SCALE = {"b": 1e9, "bn": 1e9, "billion": 1e9, "m": 1e6, "million": 1e6}


def dollar_figures(memo):
    """Every money figure in the memo, in dollars. Years and page numbers carry no $ and no unit, so they are skipped."""
    return [float(num.replace(",", "")) * _SCALE.get(unit.lower(), 1)
            for cur, num, unit in _MONEY.findall(memo) if cur or unit]


def grade(memo, ticker):
    """Step 2, an automatic test: the memo's figure must be the fiscal-year figure in the 10-K, to the nearest $0.1B.
    Returns (passed, feedback). It checks the figure and nothing else, which is a grader's limit."""
    row = FILINGS[ticker]
    found = dollar_figures(memo)
    if any(round(x / 1e8) == round(row["fiscal"] / 1e8) for x in found):
        return True, f"{billions(row['fiscal'])} matches the 10-K"
    said = f"the memo says {billions(found[0])}" if found else "the memo gives no figure"
    return False, f"{said}; the 10-K says {billions(row['fiscal'])} for {row['fy']}"

### The lessons file

`lessons.md` is a real file on Colab's disk. Each lesson is one line that starts with `- `, followed by an indented line that says which failure caused it and who approved it. `lessons_block` reads the file and puts the lessons into the system prompt of every memo. `reflect` asks the model for the rule, with the same prompt the textbook uses. `save` adds it to the file, but only when all three rules hold.

In [ ]:
LESSONS_FILE = "lessons.md"


def rules_from(text):
    """One rule per line, with any bullet, number, quote marks or code fence the model added taken off."""
    rules = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s+", "", line).strip().strip('"“”').strip()
        if line and not line.startswith(("```", "#")):
            rules.append(line)
    return rules


def read_lessons(path=LESSONS_FILE):
    """The lessons in a file: every line that starts with '- '."""
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [line[2:].strip() for line in f if line.startswith("- ")]


def lessons_block(lessons=None):
    """The system prompt for a memo: the desk's standing instruction, then lessons.md if it has anything in it.
    Pass a list to try other lessons without touching the file (section 3 does)."""
    lessons = read_lessons() if lessons is None else lessons
    if not lessons:
        return DESK
    return DESK + "\n\nLessons from past work (lessons.md):\n" + "\n".join(f"- {t}" for t in lessons)


def write_memo(ticker, lessons=None, verbose=True):
    """Step 1: the agent writes the annual revenue memo, with the lessons on the desk."""
    return agent(f"Write the annual revenue memo for {FILINGS[ticker]['name']}, in two sentences.",
                 lessons_block(lessons), verbose=verbose)


def reflect(feedback):
    """Step 3: the model turns the grader's feedback into one short rule, in its own words."""
    reply = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "You keep the research desk's lessons file. Reply with one rule of at most 20 words."},
        {"role": "user", "content": f"A revenue memo failed this check: {feedback}. Priya's note: {PRIYA} Write the rule that would have prevented it."}])
    rules = rules_from(reply.choices[0].message.content or "")
    return rules[0] if rules else PRIYA                # an empty reply: fall back to Priya's own words


def save(lesson, failure=None, approved_by=None, path=LESSONS_FILE):
    """Step 4, with the rules for the lessons file: only after a graded failure, never twice,
    and only when a person has approved it. Returns what happened, in a sentence."""
    if not failure:
        return "not saved: no graded failure behind it"
    if not approved_by:
        return "not saved: waiting for a person to approve it"
    if lesson.lower() in (l.lower() for l in read_lessons(path)):
        return f"not saved: already in {path}"
    with open(path, "a") as f:
        f.write(f"- {lesson}\n  because: {failure} · approved by {approved_by}\n")
    return f"saved to {path} (approved by {approved_by})"


def run_week(learn, approver="Priya"):
    """Four memos, Monday to Thursday. With learn=False this is chapter 1's agent with no lessons: act and hand over,
    and Priya corrects by email. With learn=True lessons.md starts empty, each memo is graded,
    and a failure becomes a lesson that every later memo reads."""
    if learn:
        with open(LESSONS_FILE, "w") as f:
            f.write("# lessons.md\n\n")
    corrections = right = 0
    for day, t in WEEK:
        print(f"{day} · {FILINGS[t]['name']} memo")
        memo = write_memo(t, lessons=None if learn else [])
        ok, feedback = grade(memo, t)
        if ok:
            right += 1
            print(f"   ✓ graded: {feedback}\n" if learn else "   ✓ right figure, nothing to correct\n")
            continue
        corrections += 1
        if not learn:
            print(f"   ✗ wrong figure: {feedback}\n   Priya corrects it: \"{PRIYA}\" The correction stays in her email.\n")
            continue
        lesson = reflect(feedback)
        print(f"   ✗ graded: {feedback}\n   reflected: \"{lesson}\"\n   {save(lesson, failure=feedback, approved_by=approver)}\n")
    print(f"Memos sent back this week: {corrections}. Right first time: {right} of {len(WEEK)}.")
    return {"corrections": corrections, "right": right}

### A week with no learning

Chapter 1's agent, with no lessons on the desk. Read the `Step 1` line of each memo: it shows which basis the model asked for, `calendar` or `fiscal`.

In [ ]:
week_without = run_week(learn=False)

### A week with learning

The same four memos, now graded, with each failure turned into a lesson that Priya approves. Every memo after a saved lesson has that lesson in its system prompt.

**If nothing fails.** A real model often asks for the fiscal year on its own, because a 10-K reports the fiscal year. Then every memo passes, nothing is reflected and nothing is saved, and `lessons.md` stays empty. That is the rule working: a lesson is added only after a graded failure. A lesson nobody needed would sit in the system prompt of every memo and cost tokens for nothing. The textbook's scripted model asks for the calendar year until it has a lesson, so its week shows the whole loop.

In [ ]:
week_with = run_week(learn=True)

print("\n--- lessons.md, on disk ---")
print(open(LESSONS_FILE).read() if read_lessons() else "lessons.md has no lessons in it. Nothing was saved this week.\n")
print("--- the system prompt for the next memo ---")
print(lessons_block())

The three rules of `save`, tried on a scratch file so they do not touch `lessons.md`:

In [ ]:
rule = "Revenue: use the fiscal year in the company's 10-K, and cite the 10-K page."
failure = "the memo says $44.9B; the 10-K says $45.7B for FY2025 (ended Nov 2, 2025)"
print("No graded failure:      ", save(rule, path="rules-demo.md"))
print("No person approved it:  ", save(rule, failure=failure, path="rules-demo.md"))
print("Approved:               ", save(rule, failure=failure, approved_by="Priya", path="rules-demo.md"))
print("The same rule again:    ", save(rule, failure=failure, approved_by="Priya", path="rules-demo.md"))
os.remove("rules-demo.md")

**Try it.**

1. Run `run_week(learn=True)` two more times. Did the model ask for the same basis each time? If it wrote a lesson, copy it here and say in one sentence whether a new analyst could follow it.
2. Run `run_week(learn=True, approver=None)`. What is in `lessons.md` afterwards, and why?
3. Force a failure: run `DESK = DESK + " Report calendar-year figures."`, then `run_week(learn=True)`, and read the lessons it saves. On Thursday the desk says calendar and the lesson says fiscal: which one did the model follow? Rerun the first cell of this section to put `DESK` back.

## Section 2 · Dreaming and the playbook (page 6.3)

### Dreaming

During the week the desk wrote notes in a hurry. Three say the same thing, two disagree, and one is out of date. On Friday, while the agent is idle, the model rereads them and writes a tidier list: it merges notes that agree, keeps the newest of notes that disagree, and drops notes that are out of date.

It writes a **new** file, `lessons-new.md`, and never edits the old one. A person compares the two and approves the new file or throws it away.

In [ ]:
# The week's notes, copied from the textbook's learning.py. Keep the two in step.
WEEK_NOTES = [
    ("Mon", "Use fiscal year (Deere)"),
    ("Tue", "Fiscal year, not calendar"),
    ("Wed", "FY figures; cite 10-K"),
    ("Wed", "Tables only for institutional clients"),
    ("Thu", "Tables whenever a client asks"),
    ("Thu", "2025 guidance: Deere expects $46-48B revenue"),
]

with open("notes-friday.md", "w") as f:
    f.write("# notes-friday.md\n\n" + "".join(f"- {day}: {text}\n" for day, text in WEEK_NOTES))
print(open("notes-friday.md").read())

In [ ]:
def dream(notes_path="notes-friday.md", new_path="lessons-new.md"):
    """Consolidate a week of notes into a NEW file. The notes file is only read, never edited."""
    with open(notes_path) as f:
        notes = f.read()
    reply = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "You tidy the research desk's notes into its lessons file. Merge notes that say the same "
                                      "thing into one rule. When two notes disagree, keep only the newer one. Drop notes that are "
                                      "out of date. Reply with the rules only, one per line."},
        {"role": "user", "content": "Today is Friday. Every company in the firm's records has now reported its 2025 results.\n\n"
                                    f"This week's notes:\n{notes}"}])
    rules = rules_from(reply.choices[0].message.content or "")
    with open(new_path, "w") as f:
        f.write(f"# {new_path} (proposed, waiting for approval)\n\n" + "".join(f"- {r}\n" for r in rules))
    with open(notes_path) as f:
        unchanged = f.read() == notes
    count = sum(line.startswith("- ") for line in notes.splitlines())
    print(f"{count} notes in, {len(rules)} rules out. {notes_path} unchanged: {unchanged}.")


dream()
print("\n--- before: notes-friday.md ---")
print(open("notes-friday.md").read())
print("--- after: lessons-new.md ---")
print(open("lessons-new.md").read())

Read the two files. Did the model merge the three fiscal-year notes into one rule? Did it keep Thursday's table note and drop Wednesday's? Did it drop the 2025 guidance, now that the results are out?

Approving is a person's job. Run the next cell only if you agree with `lessons-new.md`. If you skip it, `lessons.md` stays exactly as it was and the agent keeps reading the old lessons.

In [ ]:
# The approval: copy the new file over lessons.md. Skip this cell and nothing changes.
shutil.copy("lessons-new.md", LESSONS_FILE)
print("Approved. lessons.md is now the consolidated file. The system prompt for the next memo:\n")
print(lessons_block())

### The playbook: rewrite it whole, or edit it

A playbook is the desk's standing rules. One way to keep it current is to have the model rewrite the whole thing as a fresh, shorter summary after every task. Each summary is built from the last one, so a detail that falls out once never comes back. Watch three details: the 10-K **page** citation, the handbook clause **client-service-1**, and that Deere's year ends in **November**.

In [ ]:
# The playbook, copied from the textbook's learning.py. Keep the two in step.
PLAYBOOK = [
    "Revenue: use the fiscal year in the company's 10-K, and cite the 10-K page (Deere's year ends in early November).",
    "Recommendations: never give one in a memo, and route the client to an analyst (handbook client-service-1).",
    "Tables: use one whenever a client asks, with the source in the last column.",
]
DETAILS = ["page", "client-service-1", "November"]


def report(label, book):
    text = " ".join(book).lower()
    kept = " · ".join(f"{d}: {'kept' if d.lower() in text else 'lost'}" for d in DETAILS)
    print(f"{label:<10}{len(text.split()):>3} words. {kept}")


def rewrite_all(book):
    """One round: the model rewrites the whole playbook as a fresh, shorter summary."""
    reply = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "You keep the research desk's playbook. Rewrite the whole playbook as a fresh, "
                                      "shorter summary. Reply with the rules only, one per line."},
        {"role": "user", "content": "\n".join(book)}])
    return rules_from(reply.choices[0].message.content or "")


book = PLAYBOOK
report("Start", book)
for n in range(1, 6):
    book = rewrite_all(book)
    report(f"Round {n}", book)
print("\nAfter five rewrites:")
print("\n".join(book))

In [ ]:
# A small edit instead: add one rule and leave every other rule word for word.
edited = PLAYBOOK + ["Charts: label both axes."]
report("Edited", edited)
print("The first three rules are unchanged, word for word:", edited[:3] == PLAYBOOK)

**Try it.**

1. Run the dreaming cell again. Is `lessons-new.md` the same as the first time? Write down one difference that would stop you approving it.
2. Add a seventh note, `("Fri", "Cite the 10-K page number, not just the 10-K")`, to `WEEK_NOTES`, rerun that cell to rewrite `notes-friday.md`, and dream again. Did the model fold it into the fiscal-year rule or keep it apart?
3. Change the playbook loop to ten rounds. At which round does it stop getting shorter, and what is left of the three details?

## Section 3 · Held-out evaluation (page 6.4)

The practice week is Deere, Apple, Caterpillar and NVIDIA. Microsoft is held out: no lesson ever comes from it, and it is used only to test. A lesson that helps on Microsoft is a rule the agent learned. A lesson that helps only on the practice week is a list of memorized answers.

Three runs, each on all five memos: no lessons; memorized answers (`Deere: $45.7B`, one per practice company); and the general fiscal-year rule. The lessons are passed in directly, so `lessons.md` is not touched.

The textbook's scripted model copies a memorized answer without looking anything up. A real model may look the figure up anyway, because the desk tells it to, and may ask for the fiscal year with no lesson at all. Read the line for each memo to see what it did. This section makes about 30 model calls, so it takes a minute or two.

In [ ]:
MEMORIZED = [f"{FILINGS[t]['name']}: {billions(FILINGS[t]['fiscal'])}" for _, t in WEEK]   # answers, not a rule
RULE = "Revenue: use the fiscal year in the company's 10-K, and cite the 10-K page."         # a general rule


def evaluate(lessons, grader=grade):
    """Score the agent on the practice week and on the held-out task it never learned from.
    The held-out task is always marked by the real grader; `grader` replaces it on practice only."""
    p = h = 0
    for _, t in WEEK:
        ok, feedback = grader(write_memo(t, lessons, verbose=False), t)
        p += ok
        print(f"   practice · {FILINGS[t]['name']:<12}{'✓' if ok else '✗'} {feedback}")
    for t in HELD_OUT:
        ok, feedback = grade(write_memo(t, lessons, verbose=False), t)
        h += ok
        print(f"   held-out · {FILINGS[t]['name']:<12}{'✓' if ok else '✗'} {feedback}")
    print(f"Practice: {p} of {len(WEEK)} memos right ({p / len(WEEK):.0%}).")
    print(f"Held-out: {h} of {len(HELD_OUT)} right ({h / len(HELD_OUT):.0%}). Microsoft was never part of the practice week.\n")
    return p, h


print("No lessons:")
scores_none = evaluate([])
print("Memorized answers:", "; ".join(MEMORIZED))
scores_memorized = evaluate(MEMORIZED)
print("The general rule:", RULE)
scores_rule = evaluate([RULE])

### Reward hacking (scripted)

An agent that can edit its own tests can make them pass without getting any better. Here the grader is swapped **by hand** for one that says "pass" to every practice memo. This is scripted: no model rewrote anything. The held-out task is still marked by the real grader, which is why a locked held-out set matters.

In [ ]:
always_pass = lambda memo, ticker: (True, "looks fine")   # scripted: stands in for a grader the agent rewrote
print("No lessons, practice graded by the rewritten grader:")
scores_hacked = evaluate([], grader=always_pass)
print(f"Practice went from {scores_none[0]} to {scores_hacked[0]} of {len(WEEK)}. "
      f"Held-out was {scores_none[1]} of {len(HELD_OUT)} before and is {scores_hacked[1]} now. "
      "The held-out grade comes from the real grader, so any change there is the model's own variation, not the hack.")

**Try it.**

1. Add `"Microsoft: $281.7B"` to `MEMORIZED` and run `evaluate(MEMORIZED)`. The held-out score goes up. Say in one sentence why that score no longer tells you anything.
2. Swap the sets: make Microsoft a practice task and NVIDIA the held-out one (edit `WEEK` and `HELD_OUT`), then run `evaluate([RULE])`. Does the rule still carry over?
3. Write a stricter grader that also fails a memo with no 10-K page in it (hint: `"page" in memo`). Run `evaluate([RULE], grader=your_grader)`. Which memos fail now, and what would the lesson say?